In [1]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import cfg
from src.my_utils import set_seed, build_preproc_pipeline, cv_result, log_experiment

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

from lightgbm import LGBMClassifier

from src.features import *
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
gseed = cfg.general.seed
set_seed(gseed)

In [4]:
train_path = Path(cfg.paths.train)
submit_path = Path(cfg.paths.test)

In [5]:
df_train = pd.read_csv(train_path)
df_submit = pd.read_csv(submit_path)

In [6]:
X_train = df_train.drop(columns=["Survived"])
y_train = df_train["Survived"]

In [7]:
# skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=gseed)

# в связи с шумом и малым числом данных лучше перейти на repeated версию
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

# consistent model

Базовая модель, которая не склонна к переобучению как бустинг

In [8]:
base = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_split=10, random_state=gseed
)

In [82]:
preproc = build_preproc_pipeline(
    age_imputer=BaselineAgeImputer(),
    cabin_transformer=BaselineCabinTransformer(),
    group_transformer="passthrough",
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Embarked"],
    num_columns=["Age", "SibSp", "Parch", "Fare"],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [83]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.847762,0.005326,0.831428,0.030982,0.830527


In [84]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="consistent model",
    metrics=metrics_dict,
    note="consistent model",
)

### Age imputers

In [85]:
metrics_lst = []
for age_imputer in (
    BaselineAgeImputer(),
    ByPclassAgeImputer(),
    BySexAgeImputer(),
    BySexPclassAgeImputer(),
    ByPclassTitleAgeImputer(),
):
    preproc = build_preproc_pipeline(
        age_imputer=age_imputer,
        cabin_transformer=BaselineCabinTransformer(),
        group_transformer="passthrough",
        cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
        num_scaler="passthrough",
        cat_columns=["Pclass", "Sex", "Embarked"],
        num_columns=["Age", "SibSp", "Parch", "Fare"],
    )

    _, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
    print(metrics)
    metrics_dict = metrics.iloc[0].to_dict()
    metrics_lst.append(metrics_dict)

   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.847762       0.005326      0.831428     0.030982  0.830527
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.847762       0.005075      0.832105      0.03092  0.832772
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.849183       0.005308      0.828969     0.032438  0.830527
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD  OOF_acc
0        0.848809       0.005352      0.830984     0.031111  0.83165
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.852052       0.005133      0.832552     0.030011  0.830527


In [86]:
log_experiment(
    model_name="consistent model",
    metrics=metrics_lst[1],
    note="most effect - ByPclassAgeImputer",
)

### Title

In [87]:
preproc = build_preproc_pipeline(
    age_imputer=BaselineAgeImputer(),
    cabin_transformer=BaselineCabinTransformer(),
    group_transformer="passthrough",
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Embarked"] + ["Title"],
    num_columns=["Age", "SibSp", "Parch", "Fare"],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [88]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics  # титул бесполезен

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.8526,0.004231,0.827156,0.037338,0.82716


In [89]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="consistent model",
    metrics=metrics_dict,
    note="Title - useless",
)

### Ticket_Group_Size

In [90]:
preproc = build_preproc_pipeline(
    age_imputer=BaselineAgeImputer(),
    cabin_transformer=BaselineCabinTransformer(),
    group_transformer=GroupTransformer(),
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Embarked"],
    num_columns=["Age", "SibSp", "Parch", "Fare"] + ["Ticket_Group_Size"],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,smooth,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and

In [91]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics  # Ticket_Group_Size не очень полезна

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.850156,0.005669,0.829196,0.032404,0.828283


In [92]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="baseline model",
    metrics=metrics_dict,
    note="Ticket_Group_Size - useless",
)

### Fare, True_Fare, Log_True_Fare

In [93]:
metrics_lst = []
for col in ("Fare", "True_Fare", "Log_True_Fare"):
    preproc = build_preproc_pipeline(
        age_imputer=BaselineAgeImputer(),
        cabin_transformer=BaselineCabinTransformer(),
        group_transformer=GroupTransformer(),
        cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
        num_scaler="passthrough",
        cat_columns=["Pclass", "Sex", "Embarked"],
        num_columns=["Age", "SibSp", "Parch"] + ["Ticket_Group_Size"] + [col],
    )
    # print(preproc)
    _, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
    print(metrics)
    metrics_dict = metrics.iloc[0].to_dict()
    metrics_lst.append(metrics_dict)


   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0         0.85063       0.005348      0.828067     0.032769  0.835017
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD  OOF_acc
0         0.84634       0.005477      0.824929     0.032393  0.82716
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD  OOF_acc
0         0.84634       0.005477      0.824929     0.032393  0.82716


In [94]:
""" добавление одной из колонок сделало хуже
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.847762       0.005326      0.831428     0.030982  0.830527
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.845243        0.00459      0.823803     0.029919  0.824916
   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc
0        0.845243       0.004632      0.823803     0.029919  0.824916
"""

' добавление одной из колонок сделало хуже\n   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc\n0        0.847762       0.005326      0.831428     0.030982  0.830527\n   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc\n0        0.845243        0.00459      0.823803     0.029919  0.824916\n   TRAIN_acc_MEAN  TRAIN_acc_STD  VAL_acc_MEAN  VAL_acc_STD   OOF_acc\n0        0.845243       0.004632      0.823803     0.029919  0.824916\n'

In [95]:
log_experiment(
    model_name="consistent model",
    metrics=metrics_lst[0],
    note="most effect - Ticket_Group_Size + base fare",
)

### Ticket_Survival_Rate

In [ ]:
preproc = build_preproc_pipeline(
    age_imputer=BaselineAgeImputer(),
    cabin_transformer=BaselineCabinTransformer(),
    group_transformer=GroupTransformer(smooth=3e-4),
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Embarked"],
    num_columns=["Age", "SibSp", "Parch", "Fare"] + ["Ticket_Survival_Rate"],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,smooth,0.0003
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`

In [170]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.890785,0.006394,0.833236,0.033217,0.833895


In [174]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="consistent model",
    metrics=metrics_dict,
    note="Ticket_Survival_Rate useful with low smooth",
)

### Deck + Has_Cabin

In [ ]:
preproc = build_preproc_pipeline(
    age_imputer=BaselineAgeImputer(),
    cabin_transformer=AdvancedCabinTransformer(),
    group_transformer="passthrough",
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Embarked"],
    num_columns=["Age", "SibSp", "Parch", "Fare"],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [110]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.855269,0.004188,0.819551,0.034777,0.821549


In [111]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="consistent model",
    metrics=metrics_dict,
    note="Has_Cabin & Deck - useless",
)

### optimal features

Ticket_Group_Size + base fare + ByPclassAgeImputer + Ticket_Survival_Rate

In [ ]:
preproc = build_preproc_pipeline(
    age_imputer=ByPclassAgeImputer(),
    cabin_transformer=BaselineCabinTransformer(),
    group_transformer=GroupTransformer(smooth=1e-5),
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Embarked"],
    num_columns=[
        "Age",
        "SibSp",
        "Parch",
        "Ticket_Group_Size",
        "Fare",
        "Ticket_Survival_Rate",
    ],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,smooth,1e-05
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name``

In [172]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.890535,0.005986,0.833011,0.030079,0.836139


In [ ]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="consistent model",
    metrics=metrics_dict,
    note="all optimal features (Ticket_Group_Size + base fare + ByPclassAgeImputer + Ticket_Survival_Rate) together are actually good",
)

### random manual experiments

In [ ]:
preproc = build_preproc_pipeline(
    age_imputer=ByPclassAgeImputer(),
    cabin_transformer=AdvancedCabinTransformer(),
    group_transformer=GroupTransformer(smooth=1e-5),
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Title", "Has_Cabin"],
    num_columns=[
        "Age",
        "Ticket_Group_Size",
        "Fare",
        "Ticket_Survival_Rate",
    ],
)
preproc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dtypes', ...), ('title_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,impute_nulls,False
,smooth,1e-05
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name``

In [200]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preproc)
metrics  # один из случайных ручных экспериментов оказался дейстительно удачным

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.889463,0.007554,0.840849,0.036907,0.843996


In [201]:
metrics_dict = metrics.iloc[0].to_dict()
log_experiment(
    model_name="consistent model",
    metrics=metrics_dict,
    note="prev best + Has_Cabin",
)

In [9]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
preprocessor = Pipeline(
    [
        ("dtypes", DtypesTransformer()),
        ("title_transformer", TitleTransformer()),
        ("age_imputer", ByPclassAgeImputer()),
        ("embarked_imputer", EmbarkedImputer()),
        ("fare_imputer", FareImputer()),
        ("group_transformer", GroupTransformer(smooth=1e-6)),
        ("cabin_transformer", AdvancedCabinTransformer()),
        (
            "column_transformer",
            ColumnTransformer(
                [
                    (
                        "cat_ohe",
                        OneHotEncoder(sparse_output=False, drop="first"),
                        ["Pclass", "Sex", "Title", "Has_Cabin"],
                    ),
                    (
                        "cat_ordinal",
                        Pipeline(
                            [
                                ("binner", QuantileBinner(5)),
                                ("ordinal_encoder", OrdinalEncoder()),
                            ]
                        ),
                        ["Fare"],
                    ),
                    (
                        "num",
                        "passthrough",
                        ["Age", "Ticket_Group_Size", "Ticket_Survival_Rate"],
                    ),
                ],
                remainder="drop",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
    ]
)

In [ ]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preprocessor)
metrics  # стало хуже, бины не помогли

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.886719,0.007606,0.836579,0.037502,0.837262


В ходе экспериментов и методом случайного тыка оптимальный процесс предобработки данных
```python
preproc = build_preproc_pipeline(
    age_imputer=ByPclassAgeImputer(),
    cabin_transformer=AdvancedCabinTransformer(),
    group_transformer=GroupTransformer(smooth=1e-5),
    cat_encoder=OneHotEncoder(sparse_output=False, drop="first"),
    num_scaler="passthrough",
    cat_columns=["Pclass", "Sex", "Title", 'Has_Cabin'],
    num_columns=[
        "Age",
        "Ticket_Group_Size",
        "Fare",
        "Ticket_Survival_Rate",
    ],
)
preproc

```



In [202]:
log_dir = Path(cfg.paths.logs)
log_file_path = log_dir / "experiments.log"

df_raw = pd.read_json(log_file_path, lines=True)

df_metrics = pd.json_normalize(df_raw["metrics"])

df = pd.concat([df_raw.drop(columns=["metrics"]), df_metrics], axis=1)

df.sort_values(
    by=["OOF_acc", "VAL_acc_MEAN", "VAL_acc_STD"], ascending=[False, False, True]
)


,timestamp,model,note,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
11,2026-09-05 17:56:00,consistent model,prev best + Has_Cabin,0.889463,0.007554,0.840849,0.036907,0.843996
10,2026-09-05 17:13:28,consistent model,"best ever result, created randomly and manually",0.893304,0.005404,0.839733,0.036428,0.842873
8,2026-09-05 16:59:19,consistent model,all optimal features (Ticket_Group_Size + base...,0.890535,0.005986,0.833011,0.030079,0.836139
9,2026-09-05 17:00:03,consistent model,Ticket_Survival_Rate useful with low smooth,0.890535,0.005986,0.833011,0.030079,0.836139
7,2026-09-05 11:34:24,consistent model,all optimal features (Ticket_Group_Size + base...,0.850854,0.005395,0.829640,0.032312,0.835017
4,2026-09-05 11:15:16,consistent model,most effect - Ticket_Group_Size + base fare,0.850630,0.005348,0.828067,0.032769,0.835017
1,2026-09-05 11:13:04,consistent model,most effect - ByPclassAgeImputer,0.847762,0.005075,0.832105,0.030920,0.832772
0,2026-09-05 11:11:00,consistent model,consistent model,0.847762,0.005326,0.831428,0.030982,0.830527
5,2026-09-05 11:15:46,consistent model,Ticket_Survival_Rate useless,0.892231,0.006461,0.830991,0.033170,0.830527
3,2026-09-05 11:13:48,baseline model,Ticket_Group_Size - useless,0.850156,0.005669,0.829196,0.032404,0.828283
